# Neodragon-X × Wan2.2: distillation on a FREE Kaggle GPU

Teaches Neodragon's multistep DiT from **Wan2.2-TI2V-5B** videos (Stage A of `docs/DISTILLATION.md`), using only the free Kaggle quota (~30 GPU-hours/week).

**Setup (once):** kaggle.com → *Create → New Notebook → File → Import Notebook* (this file) → *Settings*: **Accelerator = GPU T4 x2**, **Internet = On** (needs a phone-verified account).

**How a session works** (each session ends after at most 12 h):
1. Run all cells. Part B makes teacher videos, Part C encodes them, Part D trains with a time budget and saves `lora_last.safetensors`.
2. Click **Save Version → Save & Run All (Commit)**. Kaggle keeps everything in `/kaggle/working` as the notebook *output*.
3. Next session: *Add Input → Your Work → this notebook's output*, set `PREVIOUS` below to its path, and run again. Teacher clips and the LoRA continue where they stopped.

**Honest expectations:** a few hundred teacher clips + a few thousand LoRA steps can noticeably improve motion and detail in **monolithic** mode. It does not make Neodragon equal to Wan2.2. Bringing the gain to the fast 1-1-1 mode is Stage B (not in this notebook). Stop rule: if `val_loss` does not drop below the step-0 value, stop.

> Status: the training and encoding code was smoke-tested on CPU by the author. This notebook has **not** yet been executed on a Kaggle T4. If something fails, the error message is usually enough to fix it.

In [ ]:
# ---- settings ----
BRANCH = "feat/neodragon-x-phase2"   # switch to "main" once the PRs are merged
PREVIOUS = None   # e.g. "/kaggle/input/neodragon-x-distill" (the output of your last version) to continue an earlier session
MAKE_TEACHER = True          # Part B: generate new Wan2.2 clips this session
TEACHER_CLIPS = 40           # new clips this session (~5-10 min per clip per T4; 2 GPUs work in parallel)
TEACHER_STEPS = 30           # Wan2.2 sampling steps (40 = official quality, 30 = faster)
TRAIN_HOURS = 6.0            # Part D time budget (session limit is 12 h; leave room for Parts B-C)
TRAIN_STEPS = 3000           # upper bound on optimizer steps across all sessions

In [ ]:
import os, subprocess, shutil, json, glob
os.chdir("/kaggle/working")
if not os.path.exists("neodragon"):
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/ayoub5550/neodragon.git"], check=True)
os.chdir("/kaggle/working/neodragon")
!pip -q install diffusers==0.34.0 transformers==4.46.3 accelerate==1.9.0 timm==1.0.22 einops sentencepiece imageio imageio-ffmpeg av safetensors
# carry over data + LoRA from the previous session
DATA = "/kaggle/working/data"
os.makedirs(DATA, exist_ok=True)
if PREVIOUS:
    for sub in ["teacher", "student", "stage_a"]:
        src = os.path.join(PREVIOUS, "data", sub)
        if os.path.exists(src) and not os.path.exists(os.path.join(DATA, sub)):
            shutil.copytree(src, os.path.join(DATA, sub))
    print("restored:", os.listdir(DATA))
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Part B: teacher videos with Wan2.2-TI2V-5B (Apache-2.0)
Runs in a separate virtualenv (Wan needs diffusers ≥ 0.35; Neodragon pins 0.34). One worker per T4, each taking every 2nd clip (`--shard`). Resumable: existing clips are skipped.

In [ ]:
if MAKE_TEACHER:
    if not os.path.exists("/tmp/nx/wan"):
        !python -m venv --system-site-packages /tmp/nx/wan
        !/tmp/nx/wan/bin/pip -q install "diffusers>=0.35" "transformers>=4.49" accelerate ftfy imageio imageio-ffmpeg av sentencepiece
    n_gpu = max(1, int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout))
    procs = []
    for i in range(n_gpu):
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i)}
        cmd = ["/tmp/nx/wan/bin/python", "distill/make_teacher_data.py", "--out", f"{DATA}/teacher",
               "--prompts", "distill/prompts.txt", "--clips_per_prompt", "8", "--limit", str(-(-TEACHER_CLIPS // n_gpu)),
               "--height", "480", "--width", "768", "--num_frames", "49", "--steps", str(TEACHER_STEPS),
               "--device", "cuda", "--dtype", "fp16", "--cpu_offload", "--shard", f"{i}/{n_gpu}",
               "--cache", "/tmp/nx/hf"]
        procs.append(subprocess.Popen(cmd, env=env, stdout=open(f"/kaggle/working/teacher_gpu{i}.log", "a"), stderr=subprocess.STDOUT))
    for p in procs:
        p.wait()
    print("teacher clips:", len(glob.glob(f"{DATA}/teacher/*.mp4")))
    !tail -n 3 /kaggle/working/teacher_gpu*.log

> T4 has no native bf16, so the teacher runs in fp16. If clips come out black or NaN, change `"fp16"` to `"fp32"` (slower, more memory) in the cell above.

## Part C: encode teacher clips into Neodragon latents (+ round-trip PSNR check)

In [ ]:
!python distill/encode_for_student.py --teacher {DATA}/teacher --out {DATA}/student --cache /tmp/nx/models --device cuda
print("student samples:", len(glob.glob(f"{DATA}/student/0*.pt")))

## Part D: Stage A LoRA training
Watch `val_loss`: step 0 is the original model. Lower = closer to the teacher on held-out clips. Saves every 200 steps and stops cleanly before `TRAIN_HOURS`.

In [ ]:
!python distill/train_stage_a.py --data {DATA}/student --out {DATA}/stage_a --cache /tmp/nx/models \
    --device cuda --steps {TRAIN_STEPS} --time_budget_h {TRAIN_HOURS} --resume auto

In [ ]:
import pandas as pd
log = pd.read_json(f"{DATA}/stage_a/train_log.jsonl", lines=True)
print(log.dropna(subset=["val_loss"])[["step", "val_loss"]].to_string(index=False))
log.dropna(subset=["loss"]).plot(x="step", y="loss", title="train loss")

## Part E: compare original vs distilled (monolithic mode, same seed)

In [ ]:
PROMPT = "A red fox running through fresh snow in a pine forest at sunrise"
base = (f"python scripts/inference_neodragon.py --mode monolithic --prompt '{PROMPT}' --seed 42 "
        f"--local_cache_folder /tmp/nx/models")
!{base} --output_video_folder /kaggle/working/compare/original
!{base} --lora {DATA}/stage_a/lora_last.safetensors --output_video_folder /kaggle/working/compare/distilled
from IPython.display import Video, display
for tag in ["original", "distilled"]:
    for f in sorted(glob.glob(f"/kaggle/working/compare/{tag}/*.mp4")):
        print(tag, f); display(Video(f, embed=True, width=512))

## Before you close the session
*Save Version → Save & Run All (Commit)*. Your `data/` (teacher clips, latents, LoRA) becomes this notebook's output. The model weights in `/tmp/nx` are not saved; they are downloaded again next time.